# 9 — Prvi poziv LLM-a

**Četvrtak, 14:00.** Drugi dio popodnevnog bloka.

U 13h smo YOLO zapakirali u `detect()`. Sad radimo isto s LLM-om: jedan
poziv, pa funkcija `ask()`.

Namjerno **bez SDK-a**, običnim HTTP POST-om. Poanta je da vidite da LLM API
nije magija nego JSON preko HTTP-a — to će vam trebati u 15h kad budemo
gradili workflow, i u 16h kad agent sam bude odlučivao što zvati.

Dva pitanja koja želim da odgovorite do kraja notebooka:
1. Koliko je koštalo?
2. Kako znate da je odgovor točan?

In [ ]:
# --- SETUP: pokreni ovo prvo ---  [lares-setup-v1]
# Radi i u Colabu i lokalno. Sigurno je pokrenuti vise puta.
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/hrvojenovak/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

## 1. API ključ

**Svaki od vas treba svoj ključ.** Kvota se broji po projektu, ne po ključu,
pa dijeljeni ključ znači dijeljenu kvotu.

1. Otvorite [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
2. **Create API key** → kopirajte
3. U Colabu lijevo kliknite **ključić 🔑** (Secrets)
4. Novi secret: ime `GOOGLE_API_KEY`, vrijednost = vaš ključ
5. Uključite **Notebook access**

Ako AI Studio kaže da nije dostupan: koristite **osobni** Google račun, ne
poslovni. Firmini Workspace računi često imaju to blokirano.

In [ ]:
API_KEY = None

if "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception as e:
        print("Secret nije dostupan:", type(e).__name__)
else:
    API_KEY = os.environ.get("GOOGLE_API_KEY")

if API_KEY:
    print(f"kljuc ucitan: {API_KEY[:6]}...{API_KEY[-4:]}  ({len(API_KEY)} znakova)")
else:
    print("NEMA KLJUCA — vrati se na korak 1-5 iznad.")
    print("Ne lijepi kljuc direktno u celiju: notebook ide na GitHub.")

## 2. Goli HTTP poziv

Ovo je *cijeli* API. Jedan endpoint, jedan JSON.

`contents` je razgovor. Role su `user` i `model` (kod Gemini API-ja; drugi
provideri koriste `assistant` — interface se razlikuje, ideja ne).

In [ ]:
import requests, json

MODEL = "gemini-2.5-flash"     # imena se mijenjaju; ako dobijes 404, provjeri u AI Studiju
URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

payload = {
    "contents": [
        {"role": "user", "parts": [{"text": "Objasni u jednoj rečenici što je overfitting."}]}
    ],
    "generationConfig": {"temperature": 0.0, "maxOutputTokens": 200},
}

resp = requests.post(URL, params={"key": API_KEY}, json=payload, timeout=60)
print("HTTP", resp.status_code)
data = resp.json()
print(json.dumps(data, indent=1, ensure_ascii=False)[:900])

## 3. `ask()` — interface za ljude i za kod

Isti obrazac kao `detect()` u YOLO notebooku: zamotaj u funkciju s čistim
potpisom. Dodajemo dvije stvari koje ćete stvarno trebati.

**Retry na 429.** Free tier ima ~10-15 zahtjeva u minuti. 429 nije greška,
to je normalno stanje. Backoff ima *jitter* jer nas je trinaest u sobi — bez
njega svi retryjamo u istoj sekundi i sami držimo limit probijenim.

**Brojač.** Dnevna kvota je ~250-1500 zahtjeva, ovisno o modelu. Agent u 16h
pojede 10+ zahtjeva po pokretanju, pa je dobro znati gdje ste.

In [ ]:
import time, random

STATS = {"requests": 0, "input_tokens": 0, "output_tokens": 0}


def ask(prompt: str, *, grounded: bool = False, model: str = MODEL,
        temperature: float = 0.0, max_attempts: int = 5) -> str:
    """Posalji prompt, vrati tekst. grounded=True ukljucuje Google Search."""
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    body = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": temperature, "maxOutputTokens": 1024},
    }
    if grounded:
        body["tools"] = [{"google_search": {}}]

    for attempt in range(1, max_attempts + 1):
        r = requests.post(url, params={"key": API_KEY}, json=body, timeout=90)

        if r.status_code in (429, 500, 502, 503, 504):
            if attempt == max_attempts:
                return f"[odustajem nakon {attempt} pokusaja: HTTP {r.status_code}]"
            delay = min(32, 2 ** (attempt - 1)) * (0.5 + random.random())   # jitter
            print(f"  HTTP {r.status_code}, cekam {delay:.1f}s (pokusaj {attempt})")
            time.sleep(delay)
            continue

        if r.status_code >= 400:
            return f"[greska HTTP {r.status_code}] {r.text[:300]}"

        d = r.json()
        u = d.get("usageMetadata", {})
        STATS["requests"] += 1
        STATS["input_tokens"] += u.get("promptTokenCount", 0)
        STATS["output_tokens"] += u.get("candidatesTokenCount", 0)

        cands = d.get("candidates") or []
        if not cands:
            return f"[nema odgovora - vjerojatno safety filter] {json.dumps(d)[:200]}"
        parts = (cands[0].get("content") or {}).get("parts", [])
        return "".join(p.get("text", "") for p in parts).strip()

    return "[neocekivano]"


print(ask("Nabroji tri nacina da model pretreniras. Kratko."))
print("\n", STATS)

## 4. Koliko je koštalo

Free tier ne naplaćuje, ali tokeni postoje i **na plaćenom tieru ovo je
račun**. Zato brojimo od prvog dana.

Ključna stvar koju ljudi promaše: u razgovoru se **cijela povijest šalje
ponovno u svakom pozivu**. Deset izmjena razgovora nije 10× jedan poziv,
nego kvadratno. To ćete vidjeti u 16h kad agent odradi 10 turnova.

In [ ]:
# ilustracija: cijene su za placeni tier, po milijun tokena
PRICES = {"Gemini Flash (paid)": (0.30, 2.50), "Claude Haiku 4.5": (1, 5),
          "Claude Sonnet 5": (2, 10)}

i, o = STATS["input_tokens"], STATS["output_tokens"]
print(f"do sad: {STATS['requests']} zahtjeva, {i} input + {o} output tokena\n")
for name, (pin, pout) in PRICES.items():
    print(f"  {name:22s} ${i*pin/1e6 + o*pout/1e6:.6f}")

print("\n  ...a agentska petlja od 10 turnova posalje ~69k input tokena.")

## 5. Grounding s izvorima

`ask()` vraca samo tekst i **baca izvore**. To nije dobro: grounded odgovor
bez izvora je isto tako neprovjerljiv kao ungrounded.

Kad ukljucis Google Search, Gemini uz odgovor vraca i `groundingMetadata`:
koje je upite pretrazio i koje stranice je koristio. Izvucimo to.

In [ ]:
def ask_grounded(prompt: str, model: str = MODEL) -> dict:
    """Grounded poziv koji vraca I odgovor I izvore."""
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    body = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "tools": [{"google_search": {}}],
        "generationConfig": {"temperature": 0.0, "maxOutputTokens": 1024},
    }
    r = requests.post(url, params={"key": API_KEY}, json=body, timeout=90)
    if r.status_code >= 400:
        return {"text": f"[HTTP {r.status_code}] {r.text[:200]}", "sources": [], "queries": []}

    d = r.json()
    u = d.get("usageMetadata", {})
    STATS["requests"] += 1
    STATS["input_tokens"] += u.get("promptTokenCount", 0)
    STATS["output_tokens"] += u.get("candidatesTokenCount", 0)

    cands = d.get("candidates") or []
    if not cands:
        return {"text": "[nema odgovora]", "sources": [], "queries": []}
    c = cands[0]
    text = "".join(p.get("text", "") for p in (c.get("content") or {}).get("parts", []))

    meta = c.get("groundingMetadata") or {}
    sources = []
    for ch in meta.get("groundingChunks") or []:
        w = ch.get("web") or {}
        if w:
            sources.append({"title": w.get("title", "?"), "uri": w.get("uri", "")})
    return {"text": text.strip(), "sources": sources,
            "queries": meta.get("webSearchQueries") or []}


def show(res: dict) -> None:
    print(res["text"], "\n")
    if res["queries"]:
        print("pretrazeni upiti:", res["queries"])
    if res["sources"]:
        print(f"izvori ({len(res['sources'])}):")
        for s in res["sources"]:
            print(f"  - {s['title']}")
    else:
        print("BEZ IZVORA -> model NIJE pretrazivao, odgovorio je iz tezina.")

### Isti upit, dva puta

Postavite pitanje iz **svoje** domene gdje znate tocan odgovor. Najbolje rade
pitanja o **konkretnom broju ili datumu**, jer opca pitanja model odgovori
dovoljno neodredeno da razlike ne vidite.

Tri stvari koje gledajte, ne dvije:

1. Je li ungrounded odgovor tocan?
2. Je li grounded odgovor tocan?
3. **Jesu li izvori dobri?** Ako je model nasao blog iz 2019., grounded
   odgovor je *provjerljivo* pogresan — a to je bolje od nepovjerljivo
   pogresnog, ali nije tocno.

Ako se ispise "BEZ IZVORA", model je odlucio da ne treba pretrazivati.
To je isto rezultat: grounding je *dostupan* alat, ne *obavezan* korak.

## 6. Grounded vs ungrounded — glavna poanta

Isto pitanje, dva puta. Bez alata model odgovara **iz težina** — iz onoga što
je zapamtio do svog knowledge cutoffa. S `grounded=True` dobiva Google Search
kao alat i može provjeriti.

Postavite pitanje iz **svoje** domene gdje znate točan odgovor. Najbolje
rade pitanja o **konkretnom broju ili datumu**, jer opća pitanja model
odgovori dovoljno neodređeno da razlike ne vidite.

In [ ]:
PITANJE = "Koliki je bio instalirani kapacitet vjetroelektrana u Hrvatskoj?"

print("=== BEZ alata (iz tezina) ===")
print(ask(PITANJE), "\n")

print("=== S Google Searchom (grounded) ===")
show(ask_grounded(PITANJE))

print("\n", STATS)

## 7. Za zapamtiti

**LLM API je HTTP POST s JSON-om.** Nema magije. SDK je udobnost, ne
nužnost — i zato ćete u 16h moći napisati agenta u 50 linija.

**Grounding je tool use, ne svojstvo modela.** Web search nije "opcija
modela", to je alat koji mu dodaš. Isto kao `detect()` iz 13h.

**Grounding premješta problem, ne rješava ga.** Model sad može naći *loš*
izvor i uvjereno ga citirati. Ako je ungrounded odgovor bio neodređen a
grounded konkretan — provjerite je li konkretan i *točan*.

**Tokeni se broje.** Povijest razgovora se ponovno šalje u svakom pozivu.

---

### Ako nešto ne radi

| simptom | uzrok |
|---|---|
| `NEMA KLJUCA` | secret nije dodan, ili Notebook access nije uključen |
| HTTP 400 | ime modela — provjerite u AI Studiju |
| HTTP 403 | ključ nevažeći, ili AI Studio blokiran na računu |
| HTTP 429 | rate limit — `ask()` sam retryja, samo pričekajte |
| `[nema odgovora]` | safety filter — preformulirajte prompt |